In [1]:
import fasttext as ft
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, Batch
import mlflow
from langchain_community.llms.ollama import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq
import config

class DocumentLoader:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path

    def load_documents(self):
        loader = PyPDFLoader(self.pdf_path)
        return loader.load()

class TextSplitter:
    def __init__(self, chunk_size, chunk_overlap):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            is_separator_regex=False
        )

    def split_documents(self, documents):
        return self.text_splitter.split_documents(documents)

class EmbeddingModel:
    def __init__(self, embedding_model_path):
        self.embed_model = ft.load_model(embedding_model_path)

    def get_sentence_vector(self, text):
        return self.embed_model.get_sentence_vector(text).tolist()

class DataFrameCreator:
    def __init__(self, documents, embed_model):
        self.documents = documents
        self.embed_model = embed_model

    def create_dataframe(self):
        data = []
        for doc in self.documents:
            row_data = {
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }
            data.append(row_data)

        df = pd.DataFrame(data)
        df['page_content'] = df['page_content'].replace('\\n', ' ', regex=True)
        df['id'] = range(1, len(df) + 1)
        df['payload'] = df[['page_content', 'metadata']].to_dict(orient='records')
        df['embeddings'] = df['page_content'].apply(lambda x: self.embed_model.get_sentence_vector(x))
        return df

class QdrantDatabase:
    def __init__(self, host, port, collection_name, vector_size, distance):
        self.client = QdrantClient(host=host, port=port)
        self.collection_name = collection_name
        self.vector_size = vector_size
        self.distance = distance

    def recreate_collection(self):
        try:
            self.client.delete_collection(collection_name=self.collection_name)
        except Exception as e:
            print(f"Collection '{self.collection_name}' not found. Creating a new one. Error: {e}")
        
        self.client.recreate_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=self.vector_size, distance=self.distance),
        )

class CustomRetriever:
    def __init__(self, client, embed_model, collection_name, limit):
        self.client = client
        self.embed_model = embed_model
        self.collection_name = collection_name
        self.limit = limit

    def _get_relevant_documents(self, query):
        query_vector = self.embed_model.get_sentence_vector(query).tolist()
        search_results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=self.limit
        )
        return [Document(page_content=hit.payload['page_content']) for hit in search_results]

class Chatbot:
    def __init__(self, pdf_path, embedding_model_path, host, port, collection_name, vector_size, distance, batch_size, system_prompt, model_name, num_predict, num_ctx, num_gpu, temperature, top_k, top_p, groq_api_key, selected_model):
        self.pdf_path = pdf_path
        self.embedding_model_path = embedding_model_path
        self.host = host
        self.port = port
        self.collection_name = collection_name
        self.vector_size = vector_size
        self.distance = distance
        self.batch_size = batch_size
        self.system_prompt = system_prompt
        self.model_name = model_name
        self.num_predict = num_predict
        self.num_ctx = num_ctx
        self.num_gpu = num_gpu
        self.temperature = temperature
        self.top_k = top_k
        self.top_p = top_p
        self.groq_api_key = groq_api_key
        self.selected_model = selected_model

        self.doc_loader = DocumentLoader(pdf_path)
        self.embed_model = EmbeddingModel(embedding_model_path)
        self.qdrant_db = QdrantDatabase(host, port, collection_name, vector_size, distance)

    def process_documents(self, chunk_size, chunk_overlap):
        documents = self.doc_loader.load_documents()
        text_splitter = TextSplitter(chunk_size, chunk_overlap)
        final_documents = text_splitter.split_documents(documents)
        df_creator = DataFrameCreator(final_documents, self.embed_model.embed_model)
        self.df = df_creator.create_dataframe()
        self.qdrant_db.recreate_collection()
        self.qdrant_db.upsert_embeddings(self.df, self.batch_size)

    def create_retriever(self, limit):
        self.retriever = CustomRetriever(self.qdrant_db.client, self.embed_model.embed_model, self.collection_name, limit)

    def create_llm_chain(self):
        llm = Ollama(
            model=self.model_name,
            num_predict=self.num_predict,
            num_ctx=self.num_ctx,
            num_gpu=self.num_gpu,
            temperature=self.temperature,
            top_k=self.top_k,
            top_p=self.top_p
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", "{input}"),
        ])
        question_answer_chain = create_stuff_documents_chain(llm, prompt)
        self.chain = create_retrieval_chain(self.retriever, question_answer_chain)

    def create_groq_chain(self, limit):
        llm = ChatGroq(groq_api_key=self.groq_api_key, model_name=self.selected_model)
        prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", "{input}"),
        ])
        question_answer_chain = create_stuff_documents_chain(llm, prompt)
        self.chain = create_retrieval_chain(self.retriever, question_answer_chain)

    def generate_response(self, query):
        response = self.chain.invoke({"input": query})
        return response

# Example usage
if __name__ == "__main__":
    pdf_path = '/Users/nitastha/Desktop/Fundamental Rules Hindi - new.pdf'
    embedding_model_path = '/Users/nitastha/Desktop/NitishFiles/Projects/wiki.hi/wiki.hi.bin'
    host = 'localhost'
    port = 6333
    collection_name = "my_collection"
    vector_size = 300
    distance = Distance.COSINE
    batch_size = 4000
    system_prompt = (
        """<s>[INST] आप एक सम्मानीय सहायक हैं। आपका काम नीचे दिए गए संदर्भ से प्रश्नों का उत्तर देना है। आप केवल हिंदी भाषा में उत्तर दे सकते हैं। धन्यवाद।
        You are never ever going to generate responses in English. You are always going to generate responses in Hindi no matter what. You also need to keep your answer short and to the point.

        संदर्भ: {context} </s>
        """
    )
    model_name = 'llama3'
    num_predict = 100
    num_ctx = 3000
    num_gpu = 2
    temperature = 0.7
    top_k = 50
    top_p = 0.95
    groq_api_key = config.GROQ_API_KEY
    selected_model = "llama3-groq-70b-8192-tool-use-preview"
    limit = 50

    chatbot = Chatbot(
        pdf_path,
        embedding_model_path,
        host,
        port,
        collection_name,
        vector_size,
        distance,
        batch_size,
        system_prompt,
        model_name,
        num_predict,
        num_ctx,
        num_gpu,
        temperature,
        top_k,
        top_p,
        groq_api_key,
        selected_model
    )

    

In [2]:
# Process documents with specified chunk size and overlap
chatbot.process_documents(chunk_size=300, chunk_overlap=50)

# Create retriever with specified limit
chatbot.create_retriever(limit=30)

# # Create LLM chain
# chatbot.create_llm_chain()

# # Generate response using the LLM chain
# query = 'प्रारंभिक वेतन निर्धारण करने का अधिकार किसे है?'
# response = chatbot.generate_response(query)
# print(response)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)


AssertionError: Unknown arguments: ['ignore_errors']

In [ ]:


# Create Groq chain
chatbot.create_groq_chain(limit=limit)

# Generate response using the Groq chain
query = 'राज्य सचिव आदेश 1 क्या है?'
response = chatbot.generate_response(query)
print(response)